# Setup

In [7]:
import json, os, re
from pathlib import Path
from typing import Literal

import anthropic
import jinja2
import yaml
from dotenv import load_dotenv
from pydantic import BaseModel

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

load_dotenv(REPO / ".env", override=True)

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "No ANTHROPIC_API_KEY in the environment. "
        "Copy .env.example to .env, set the key, then re-run this cell."
    )

MODEL = "claude-sonnet-5"

client = anthropic.Anthropic()

print(f"model: {MODEL}")

model: claude-sonnet-5


# The rubric, the prompt, and the problem


In [ ]:
PHASE = yaml.safe_load((REPO / "prompts/phases/problem_definition.v3.yaml").read_text())
PROBLEM = yaml.safe_load((REPO / "cases/problems/grade_average.yaml").read_text())

# Criterion text and guidance live in criteria/catalog.yaml. That file is the authority
# for them and it migrates into the SvelteKit product, so nothing here may restate it --
# this cell reads the catalog. A phase prompt names only ids; the text comes from here,
# which is what keeps the rubric and the prompt from drifting apart.
CATALOG = {
    c["id"]: c
    for c in yaml.safe_load((REPO / "criteria/catalog.yaml").read_text())["criteria"]
}


def criteria_for(phase: dict) -> list[dict]:
    """The catalog entries this phase prompt asks to have judged, in the prompt's order."""
    judged = phase["criteria"]["judged"]
    missing = [cid for cid in judged if cid not in CATALOG]
    if missing:
        raise KeyError(f"listed in {phase['id']} but absent from criteria/catalog.yaml: {missing}")
    astray = [cid for cid in judged if CATALOG[cid]["phase"] != phase["phase"]]
    if astray:
        raise ValueError(f"{phase['id']} judges criteria belonging to another phase: {astray}")
    return [CATALOG[cid] for cid in judged]


CRITERIA = criteria_for(PHASE)

# The system prompt is the shared preamble plus this phase's own section. Anything true of
# every phase -- the three verdicts, naming-without-supplying, student text being data --
# lives in the preamble and is never repeated in a phase file.
PREAMBLE = (REPO / "prompts" / PHASE["compose"]["preamble"]).read_text()
SYSTEM = f"{PREAMBLE}\n\n---\n\n{PHASE['system']}"

print(f"{PHASE['id']} v{PHASE['version']} ({PHASE['status']})\n")
for c in CRITERIA:
    print(f"  {c['id']:28} {c['gate']:9} {c['text']}")


## 2. Verdict


In [ ]:
class Verdict(BaseModel):
    criterion_id: str
    verdict: Literal["met", "unmet", "uncertain"]
    evidence: str
    confidence: Literal["low", "medium", "high"]


class JudgeResult(BaseModel):
    verdicts: list[Verdict]


# Mirrors prompts/base/output_schema.v2.json -- keep the two in step.
#
# Three verdicts, not two: `uncertain` is for an artifact that does not settle the
# question. It blocks advancement exactly as `unmet` does, but it reaches the student as a
# question about what they wrote rather than as a correction (FR-VER-01 to 03).
#
# Note what is absent: there is no overall pass/fail field. The judge returns per-criterion
# verdicts and aggregation decides advancement (FR-VER-05). Adding one here is a defect.
print(json.dumps(JudgeResult.model_json_schema(), indent=2)[:400], "...")


## 3. Rendering the prompt

In [ ]:
_jinja = jinja2.Environment(
    trim_blocks=True, lstrip_blocks=True, undefined=jinja2.StrictUndefined
)


def render_user_prompt(
    problem: dict, artifact: dict, criteria: list[dict], attempt: int = 1,
    prior: dict | None = None,
) -> str:
    """Render the phase template.

    `artifact` is the student's submission for this phase. `prior` holds the artifacts they
    submitted in earlier phases, which the later phases are judged against -- Cases against
    their Problem Definition, Design of the Algorithm against both. Problem Definition, being
    first, ignores it.
    """
    return _jinja.from_string(PHASE["user_template"]).render(
        problem=problem,
        artifact=artifact,
        criteria_to_judge=criteria,
        attempt=attempt,
        prior=prior or {},
    )


good_artifact = {
    "summary": (
        "I need to work out the class's mean score. The program reads how many students "
        "there are, then reads each student's mark, adds them up and divides by the count."
    ),
    "inputs": "int n (the student count), then n floats, one score per student",
    "outputs": "a single float: the mean of the n scores, printed to two decimal places",
    "constraints": "n is at least 1, and every score is between 0 and 100 inclusive",
}

prompt = render_user_prompt(PROBLEM, good_artifact, CRITERIA)
print(prompt)


## 4. The judge call

One call judges every criterion at once. `output_format=JudgeResult` is what turns the reply into
a validated object instead of text — `resp.parsed_output` comes back as a `JudgeResult`, already
type-checked.

Note what is *absent*: no `temperature`. Deliberation is bought with `effort` and adaptive
thinking instead, which lets the model reason before committing to a verdict.

In [ ]:
def judge(
    problem: dict, artifact: dict, criteria: list[dict], attempt: int = 1,
    prior: dict | None = None,
) -> JudgeResult:
    """Judge one submission against every criterion in a single call."""
    try:
        resp = client.messages.parse(
            model=MODEL,
            max_tokens=PHASE["model"]["max_output_tokens"],
            system=SYSTEM,
            messages=[
                {
                    "role": "user",
                    "content": render_user_prompt(
                        problem, artifact, criteria, attempt, prior
                    ),
                }
            ],
            thinking={"type": "adaptive"},
            output_config={"effort": PHASE["model"]["effort"]},
            output_format=JudgeResult,
        )
    except anthropic.AuthenticationError:
        raise RuntimeError(
            "No valid API key. Copy .env.example to .env and set ANTHROPIC_API_KEY."
        ) from None
    except anthropic.RateLimitError as e:
        retry_after = e.response.headers.get("retry-after", "60")
        raise RuntimeError(f"Rate limited; retry after {retry_after}s.") from None
    except anthropic.APIStatusError as e:
        raise RuntimeError(f"API error {e.status_code}: {e.message}") from None
    except anthropic.APIConnectionError:
        raise RuntimeError("Could not reach the API. Check your connection.") from None

    if resp.stop_reason == "max_tokens":
        raise RuntimeError(
            "Response hit max_tokens and the verdict list is truncated. "
            "Raise max_output_tokens in the phase YAML, or lower `effort`."
        )
    return resp.parsed_output


result = judge(PROBLEM, good_artifact, CRITERIA)

for v in result.verdicts:
    print(f"{v.verdict:9}  {v.criterion_id:28} ({v.confidence})")
    print(f"        evidence: {v.evidence!r}\n")